# Delta Lakehouse Exploration

This notebook demonstrates how to work with the Delta Lakehouse pipeline framework.

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

from pipelines.utils.spark_session import create_spark_session
from pipelines.sample_etl_pipeline import SampleETLPipeline
from pipelines.utils.delta_operations import DeltaOperations
from delta.tables import DeltaTable

In [ ]:
# Create Spark session
spark = create_spark_session()
print("Spark session created successfully!")

## Run the Sample Pipeline

In [ ]:
# Run the sample ETL pipeline
pipeline = SampleETLPipeline(spark)
pipeline.run()
print("Pipeline completed successfully!")

## Explore the Data Layers

In [ ]:
# Bronze Layer - Raw Data
print("=== BRONZE LAYER ===")
bronze_df = spark.read.format("delta").load("../data/bronze/employees")
bronze_df.show()
print(f"Records: {bronze_df.count()}")

In [ ]:
# Silver Layer - Cleaned Data
print("=== SILVER LAYER ===")
silver_df = spark.read.format("delta").load("../data/silver/employees")
silver_df.show()
print(f"Records: {silver_df.count()}")

In [ ]:
# Gold Layer - Analytics
print("=== GOLD LAYER - Department Stats ===")
dept_stats = spark.read.format("delta").load("../data/gold/department_stats")
dept_stats.show()

print("=== GOLD LAYER - Salary Band Stats ===")
salary_stats = spark.read.format("delta").load("../data/gold/salary_band_stats")
salary_stats.show()

## Delta Lake Features

In [ ]:
# Table History
print("=== TABLE HISTORY ===")
bronze_table = DeltaTable.forPath(spark, "../data/bronze/employees")
history = bronze_table.history()
history.select("version", "timestamp", "operation").show()

In [ ]:
# Time Travel - Query previous version
print("=== TIME TRAVEL ===")
# Query version 0 (if exists)
try:
    historical_df = spark.read.format("delta").option("versionAsOf", 0).load("../data/silver/employees")
    print("Historical data (version 0):")
    historical_df.show()
except:
    print("No historical versions available yet")

## Data Visualization

In [ ]:
# Convert to Pandas for visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Get department stats
dept_stats_pd = dept_stats.toPandas()

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Employee count by department
axes[0, 0].bar(dept_stats_pd['department'], dept_stats_pd['employee_count'])
axes[0, 0].set_title('Employee Count by Department')
axes[0, 0].set_xlabel('Department')
axes[0, 0].set_ylabel('Count')

# Average salary by department
axes[0, 1].bar(dept_stats_pd['department'], dept_stats_pd['avg_salary'])
axes[0, 1].set_title('Average Salary by Department')
axes[0, 1].set_xlabel('Department')
axes[0, 1].set_ylabel('Average Salary')

# Salary band distribution
salary_band_pd = salary_stats.toPandas()
axes[1, 0].pie(salary_band_pd['employee_count'], labels=salary_band_pd['salary_band'], autopct='%1.1f%%')
axes[1, 0].set_title('Salary Band Distribution')

# Employee salary scatter
silver_pd = silver_df.select('name', 'department', 'salary').toPandas()
for dept in silver_pd['department'].unique():
    dept_data = silver_pd[silver_pd['department'] == dept]
    axes[1, 1].scatter(range(len(dept_data)), dept_data['salary'], label=dept)
axes[1, 1].set_title('Employee Salaries by Department')
axes[1, 1].set_xlabel('Employee Index')
axes[1, 1].set_ylabel('Salary')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Custom Queries

In [ ]:
# Register table for SQL queries
silver_df.createOrReplaceTempView("employees")

# Run SQL queries
result = spark.sql("""
    SELECT 
        department,
        COUNT(*) as employee_count,
        AVG(salary) as avg_salary,
        MAX(salary) as max_salary
    FROM employees 
    GROUP BY department
    ORDER BY avg_salary DESC
""")

print("=== SQL QUERY RESULTS ===")
result.show()

In [ ]:
# Advanced analytics
analytics = spark.sql("""
    SELECT 
        salary_band,
        department,
        COUNT(*) as count,
        AVG(years_employed) as avg_tenure
    FROM employees
    GROUP BY salary_band, department
    ORDER BY salary_band, department
""")

print("=== SALARY BAND vs DEPARTMENT ANALYSIS ===")
analytics.show()

In [ ]:
# Don't forget to stop Spark session
spark.stop()
print("Spark session stopped.")